# JSON Corpus Processing Notebook

本 notebook 用于处理 Open RAG Benchmark 已提供的 JSON 格式论文语料。

主要目标：
1. 读取 `corpus/*.json` 论文文件；
2. 对 abstract 和 sections 做文本清洗；
3. 构建三种 chunking 策略；
4. 输出清洗后的 section 文件与三种 chunk 文件；
5. 统计不同 chunking 的数量与长度分布。

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from statistics import mean

# 为了保证 notebook 可以直接运行，这里直接写入辅助函数，避免依赖外部脚本导入失败。


## 1. 文本清洗函数

这部分负责：
- 去除页码与 arXiv 噪声；
- 修复断行连字符；
- 合并不必要的单行换行；
- 标准化空白字符；
- 辅助识别 references/bibliography。

In [2]:
import re
from typing import Iterable

REFERENCE_HEADING_PATTERN = re.compile(r"^(references|bibliography|acknowledg(e)?ments?|appendix)$", re.IGNORECASE)
ARXIV_NOISE_PATTERN = re.compile(r"arxiv:\\s*\\d{4}\\.\\d{4,5}(v\\d+)?", re.IGNORECASE)
PAGE_NUMBER_PATTERN = re.compile(r"^\\s*\\d+\\s*$")
WHITESPACE_PATTERN = re.compile(r"[ \\t]+")
MULTI_NEWLINE_PATTERN = re.compile(r"\\n{3,}")
INLINE_NEWLINE_PATTERN = re.compile(r"(?<!\\n)\\n(?!\\n)")
HYPHEN_BREAK_PATTERN = re.compile(r"(\\w+)-\\n(\\w+)")

def normalize_whitespace(text: str) -> str:
    text = text.replace('\xa0', ' ')
    text = WHITESPACE_PATTERN.sub(' ', text)
    text = MULTI_NEWLINE_PATTERN.sub('\n\n', text)
    return text.strip()

def fix_hyphenation(text: str) -> str:
    return HYPHEN_BREAK_PATTERN.sub(r'\1\2', text)

def merge_inline_newlines(text: str) -> str:
    return INLINE_NEWLINE_PATTERN.sub(' ', text)

def remove_noise_lines(text: str) -> str:
    cleaned_lines = []
    for line in text.splitlines():
        stripped = line.strip()
        if PAGE_NUMBER_PATTERN.fullmatch(stripped):
            continue
        if ARXIV_NOISE_PATTERN.search(stripped):
            continue
        cleaned_lines.append(line)
    return '\n'.join(cleaned_lines)

def is_reference_heading(text: str) -> bool:
    candidate = normalize_whitespace(text).strip(':. ')
    return bool(REFERENCE_HEADING_PATTERN.fullmatch(candidate))

def clean_text(text: str, merge_lines: bool = False) -> str:
    text = fix_hyphenation(text)
    text = remove_noise_lines(text)
    if merge_lines:
        text = merge_inline_newlines(text)
    text = normalize_whitespace(text)
    return text

def first_reference_index(section_texts: Iterable[str]) -> int | None:
    for idx, text in enumerate(section_texts):
        if is_reference_heading(text):
            return idx
    return None


## 2. 三种 chunking 策略

这里实现三种切块方法：
- Fixed chunking：固定长度切块；
- Recursive chunking：递归式切块；
- Section-aware chunking：优先保留 section 结构，仅对长 section 再切块。

In [3]:
from dataclasses import dataclass

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    RecursiveCharacterTextSplitter = None

@dataclass
class ChunkRecord:
    chunk_id: str
    doc_id: str
    section_id: int
    chunk_index: int
    strategy: str
    text: str
    title: str
    metadata: dict

class FixedChunker:
    def __init__(self, chunk_size: int = 1200, chunk_overlap: int = 150):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

    def split(self, text: str) -> list[str]:
        chunks = []
        start = 0
        while start < len(text):
            end = min(len(text), start + self.chunk_size)
            chunks.append(text[start:end])
            if end >= len(text):
                break
            start = max(0, end - self.chunk_overlap)
        return chunks

class RecursiveChunker:
    def __init__(self, chunk_size: int = 1200, chunk_overlap: int = 150):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        if RecursiveCharacterTextSplitter is None:
            self._splitter = None
        else:
            self._splitter = RecursiveCharacterTextSplitter(
                chunk_size=chunk_size,
                chunk_overlap=chunk_overlap,
                separators=['\n\n', '\n', '. ', ' ', '']
            )

    def split(self, text: str) -> list[str]:
        if self._splitter is None:
            return FixedChunker(self.chunk_size, self.chunk_overlap).split(text)
        return self._splitter.split_text(text)

class SectionAwareChunker:
    def __init__(self, max_section_chars: int = 1600, chunk_size: int = 1200, chunk_overlap: int = 150):
        self.max_section_chars = max_section_chars
        self.recursive_chunker = RecursiveChunker(chunk_size=chunk_size, chunk_overlap=chunk_overlap)

    def split(self, text: str) -> list[str]:
        if len(text) <= self.max_section_chars:
            return [text]
        return self.recursive_chunker.split(text)

def build_chunk_records(doc_id: str, section_id: int, title: str, strategy: str, chunks, extra_metadata=None):
    records = []
    extra_metadata = extra_metadata or {}
    for idx, chunk_text in enumerate(chunks):
        records.append({
            'chunk_id': f'{doc_id}_sec{section_id}_chunk{idx}_{strategy}',
            'doc_id': doc_id,
            'section_id': section_id,
            'chunk_index': idx,
            'strategy': strategy,
            'title': title,
            'text': chunk_text,
            'metadata': extra_metadata,
        })
    return records


## 3. 定义数据路径

请确保你的 JSON 语料目录已经下载完成。

In [4]:
DATA_ROOT = Path('project/data/open_ragbench/pdf/arxiv')
CORPUS_DIR = DATA_ROOT / 'corpus'
OUTPUT_DIR = Path('project/data/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLEANED_PATH = OUTPUT_DIR / 'cleaned_sections.jsonl'
FIXED_PATH = OUTPUT_DIR / 'chunks_fixed.jsonl'
RECURSIVE_PATH = OUTPUT_DIR / 'chunks_recursive.jsonl'
SECTION_AWARE_PATH = OUTPUT_DIR / 'chunks_section_aware.jsonl'
STATS_PATH = OUTPUT_DIR / 'chunk_stats.json'

fixed_chunker = FixedChunker(chunk_size=1200, chunk_overlap=150)
recursive_chunker = RecursiveChunker(chunk_size=1200, chunk_overlap=150)
section_aware_chunker = SectionAwareChunker(max_section_chars=1600, chunk_size=1200, chunk_overlap=150)


## 4. 读取 JSON 并提取清洗后的 sections

In [5]:
def iter_corpus_files():
    return sorted(CORPUS_DIR.glob('*.json'))

def load_json(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def dump_jsonl(path: Path, records):
    with open(path, 'w', encoding='utf-8') as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + '\n')

def extract_cleaned_sections(doc: dict):
    title = doc.get('title', '')
    doc_id = doc.get('id', '')
    sections = doc.get('sections', [])

    section_texts = [section.get('text', '') for section in sections]
    ref_idx = first_reference_index(section_texts)
    if ref_idx is not None:
        sections = sections[:ref_idx]

    cleaned = []

    abstract = clean_text(doc.get('abstract', ''), merge_lines=True)
    if len(abstract) >= 50:
        cleaned.append({
            'doc_id': doc_id,
            'section_id': -1,
            'section_label': 'abstract',
            'title': title,
            'text': abstract,
            'categories': doc.get('categories', []),
            'source': 'open_ragbench_json'
        })

    for idx, section in enumerate(sections):
        text = clean_text(section.get('text', ''), merge_lines=True)
        if len(text) < 50:
            continue
        cleaned.append({
            'doc_id': doc_id,
            'section_id': idx,
            'section_label': 'section',
            'title': title,
            'text': text,
            'categories': doc.get('categories', []),
            'source': 'open_ragbench_json'
        })

    return cleaned


## 5. 处理全部论文，生成 cleaned sections

In [6]:
cleaned_sections = []
for file_path in iter_corpus_files():
    doc = load_json(file_path)
    cleaned_sections.extend(extract_cleaned_sections(doc))

dump_jsonl(CLEANED_PATH, cleaned_sections)
print(f'Cleaned sections saved to: {CLEANED_PATH}')
print(f'Total cleaned sections: {len(cleaned_sections)}')
cleaned_sections[:2]


Cleaned sections saved to: project\data\processed\cleaned_sections.jsonl
Total cleaned sections: 0


[]

## 6. 构建三种 chunk 文件

In [7]:
def build_chunks(cleaned_sections, strategy: str):
    all_chunks = []
    for section in cleaned_sections:
        text = section['text']

        if strategy == 'fixed':
            split_chunks = fixed_chunker.split(text)
        elif strategy == 'recursive':
            split_chunks = recursive_chunker.split(text)
        elif strategy == 'section_aware':
            split_chunks = section_aware_chunker.split(text)
        else:
            raise ValueError(f'Unknown strategy: {strategy}')

        records = build_chunk_records(
            doc_id=section['doc_id'],
            section_id=section['section_id'],
            title=section['title'],
            strategy=strategy,
            chunks=split_chunks,
            extra_metadata={
                'section_label': section['section_label'],
                'categories': section['categories'],
                'source': section['source'],
            }
        )
        all_chunks.extend(records)
    return all_chunks

fixed_chunks = build_chunks(cleaned_sections, 'fixed')
recursive_chunks = build_chunks(cleaned_sections, 'recursive')
section_aware_chunks = build_chunks(cleaned_sections, 'section_aware')

dump_jsonl(FIXED_PATH, fixed_chunks)
dump_jsonl(RECURSIVE_PATH, recursive_chunks)
dump_jsonl(SECTION_AWARE_PATH, section_aware_chunks)

print(f'Fixed chunks saved to: {FIXED_PATH}')
print(f'Recursive chunks saved to: {RECURSIVE_PATH}')
print(f'Section-aware chunks saved to: {SECTION_AWARE_PATH}')


Fixed chunks saved to: project\data\processed\chunks_fixed.jsonl
Recursive chunks saved to: project\data\processed\chunks_recursive.jsonl
Section-aware chunks saved to: project\data\processed\chunks_section_aware.jsonl


## 7. 统计三种 chunking 的表现

In [8]:
def summarize_chunks(chunks):
    lengths = [len(item['text']) for item in chunks]
    return {
        'num_chunks': len(chunks),
        'avg_chunk_chars': round(mean(lengths), 2) if lengths else 0,
        'max_chunk_chars': max(lengths) if lengths else 0,
        'min_chunk_chars': min(lengths) if lengths else 0,
    }

stats = {
    'cleaned_sections': len(cleaned_sections),
    'fixed': summarize_chunks(fixed_chunks),
    'recursive': summarize_chunks(recursive_chunks),
    'section_aware': summarize_chunks(section_aware_chunks),
}

with open(STATS_PATH, 'w', encoding='utf-8') as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)

print(json.dumps(stats, ensure_ascii=False, indent=2))


{
  "cleaned_sections": 0,
  "fixed": {
    "num_chunks": 0,
    "avg_chunk_chars": 0,
    "max_chunk_chars": 0,
    "min_chunk_chars": 0
  },
  "recursive": {
    "num_chunks": 0,
    "avg_chunk_chars": 0,
    "max_chunk_chars": 0,
    "min_chunk_chars": 0
  },
  "section_aware": {
    "num_chunks": 0,
    "avg_chunk_chars": 0,
    "max_chunk_chars": 0,
    "min_chunk_chars": 0
  }
}
